In [1]:
import numpy as np
import random

# 상태 공간: 에이전트가 있을 수 있는 위치 (0~5번 칸)
state_space = [0, 1, 2, 3, 4, 5]

# 행동 공간: -1(왼쪽 이동), 1(오른쪽 이동)
action_space = [-1, 1]

# Q-table 초기화: (행=상태 수, 열=행동 수) → 전부 0으로 시작
# Q[state][action_index] = 특정 상태에서 특정 행동을 했을 때의 기대 가치
Q = np.zeros((len(state_space), len(action_space)))
print(Q)

[[0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]
 [0. 0.]]


In [2]:
alpha = 0.1          # 학습률: 새로운 정보를 기존 Q값에 얼마나 반영할지
                     # α=0이면 학습 안 함, α=1이면 새 값으로 완전 교체
gamma = 0.9          # 할인율: 미래 보상을 얼마나 중요하게 볼지
                     # γ→1이면 먼 미래도 중요, γ→0이면 즉각 보상만 중시
epsilon = 1.0        # 탐험 확률: 처음엔 100% 랜덤 행동
epsilon_decay = 0.99 # 매 스텝마다 epsilon을 0.99배씩 줄임 (Decay ε-Greedy)
epsilon_min = 0.1    # 탐험 확률 최솟값: 최소 10%는 탐험 유지
episodes = 500       # 전체 학습 반복 횟수

# 보상 함수: state 4에 도달하면 +10, 그 외엔 0
def get_reward(state):
    return 10 if state == 4 else 0

In [3]:
for episode in range(episodes):
    state = 0  # 매 에피소드마다 시작 위치 0으로 초기화

    for step in range(20):  # 한 에피소드에서 최대 20번 이동

        # ── ① 행동 선택 (ε-Greedy) ──────────────────────────────
        if random.random() < epsilon:
            action_index = random.randint(0, 1)  # 탐험: 랜덤 행동
        else:
            action_index = np.argmax(Q[state])   # 이용: Q값이 가장 높은 행동 선택

        action = action_space[action_index]  # action_index → 실제 이동값(-1 or 1)으로 변환

        # ── ② 다음 상태 계산 ────────────────────────────────────
        next_state = state + action  # 현재 위치에서 행동만큼 이동

        # 상태 공간 벗어나면 제자리 유지 (0~4 범위)
        if next_state < 0 or next_state > 4:
            next_state = state

        # ── ③ 보상 수령 ─────────────────────────────────────────
        reward = get_reward(next_state)  # state 4이면 +10, 아니면 0

        # ── ④ Q값 갱신 (벨만 최적 방정식) ───────────────────────
        old_q = Q[state][action_index]           # 현재 Q값
        next_max = np.max(Q[next_state])         # 다음 상태에서 가장 높은 Q값

        # Q-learning 업데이트 수식 (off-policy)
        # Q(s,a) ← Q(s,a) + α * (R + γ * max Q(s') - Q(s,a))
        Q[state][action_index] = old_q + alpha * (reward + gamma * next_max - old_q)

        # ── ⑤ 상태 이동 ─────────────────────────────────────────
        state = next_state

        # 목표 도달 시 에피소드 종료
        if reward == 10:
            break

        # ε 감소: 학습이 진행될수록 탐험 줄이고 이용 늘림
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

print(Q)

[[ 6.12611616  7.29      ]
 [ 6.27875745  8.1       ]
 [ 6.80578178  9.        ]
 [ 7.37490273 10.        ]
 [ 0.          0.        ]
 [ 0.          0.        ]]
